In [2]:
import os
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [3]:
load_dotenv()

db_uri = os.getenv("DB_URI")
engine = create_engine(db_uri)

print("Database connection established successfully.", db_uri is not None)

Database connection established successfully. True


In [4]:
# فحص الأسماء الحقيقية لأعمدة جدول التعبئات في قاعدة البيانات
print(pd.read_sql("SELECT * FROM internet_recharges LIMIT 1;", con=engine).columns.tolist())

['tx_id', 'subscriber_id', 'timestamp', 'amount_dzd', 'payment_method', 'tx_status', 'days_granted']


In [5]:
query_recharges = """
SELECT 
    subscriber_id,
    COUNT(*) AS total_tx_attempts,
    COUNT(*) FILTER (WHERE tx_status = 'SUCCESS') AS successful_recharges,
    COUNT(*) FILTER (WHERE tx_status != 'SUCCESS') AS failed_recharges,
    COUNT(*) FILTER (WHERE payment_method IN ('Edahabia_Online', 'CIB_Online')) AS online_payment_attempts,
    SUM(amount_dzd) AS total_recharged_dzd,
    SUM(days_granted) AS total_days_granted,
    MAX(timestamp) FILTER (WHERE tx_status = 'SUCCESS') AS last_successful_recharge_date
FROM internet_recharges
GROUP BY subscriber_id
ORDER BY subscriber_id
LIMIT 10;
"""

df_rech_summary = pd.read_sql(query_recharges, con=engine)
df_rech_summary

,subscriber_id,total_tx_attempts,successful_recharges,failed_recharges,online_payment_attempts,total_recharged_dzd,total_days_granted,last_successful_recharge_date
0,SUB-10000,6,6,0,1,12000,180,2025-07-27 21:01:00
1,SUB-10001,13,12,1,13,24000,360,2025-12-17 08:08:00
2,SUB-10002,6,6,0,1,15600,180,2025-08-10 12:48:00
3,SUB-10003,12,11,1,9,22000,330,2025-12-01 22:49:00
4,SUB-10004,11,11,0,1,28600,330,2025-12-09 11:48:00
5,SUB-10005,13,12,1,10,26400,360,2025-12-22 13:45:00
6,SUB-10006,15,12,3,14,36000,360,2025-12-25 17:44:00
7,SUB-10007,12,12,0,11,31200,360,2025-12-06 12:37:00
8,SUB-10008,11,11,0,2,24200,330,2025-12-04 08:29:00
9,SUB-10009,6,6,0,0,25200,180,2025-08-10 11:48:00


In [6]:
# فحص الأسماء الحقيقية لأعمدة جدول التعبئات في قاعدة البيانات
print(pd.read_sql("SELECT * FROM technical_tickets LIMIT 1;", con=engine).columns.tolist())

['ticket_id', 'subscriber_id', 'opened_at', 'closed_at', 'issue_category', 'line_attenuation_db']


In [7]:
query_tickets = """
SELECT 
    subscriber_id,
    COUNT(*) AS total_tickets,
    COUNT(*) FILTER (WHERE closed_at IS NULL) AS unresolved_tickets,
    ROUND(AVG(line_attenuation_db)::numeric, 2) AS avg_attenuation_db,
    MAX(line_attenuation_db) AS max_attenuation_db,
    ROUND(AVG(EXTRACT(EPOCH FROM (closed_at - opened_at)) / 3600.0)::numeric, 2) AS avg_resolution_hours
FROM technical_tickets
GROUP BY subscriber_id
ORDER BY subscriber_id
LIMIT 10;
"""

df_tkt_summary = pd.read_sql(query_tickets, con=engine)
df_tkt_summary

,subscriber_id,total_tickets,unresolved_tickets,avg_attenuation_db,max_attenuation_db,avg_resolution_hours
0,SUB-10000,1,0,20.70,20.7,78.08
1,SUB-10002,2,0,17.50,18.5,42.99
2,SUB-10005,3,0,29.73,33.2,103.64
3,SUB-10009,2,0,15.80,16.4,121.26
4,SUB-10010,2,0,18.35,18.4,107.28
5,SUB-10012,2,0,14.50,16.0,124.16
6,SUB-10013,4,1,32.18,35.5,94.28
7,SUB-10016,5,1,31.12,34.0,59.32
8,SUB-10019,1,0,18.50,18.5,26.28
9,SUB-10020,1,0,22.70,22.7,41.07


In [8]:
query_fixe = """
SELECT 
    landline_number,
    COUNT(*) AS total_fixe_bills,
    COUNT(*) FILTER (WHERE payment_status = 'LATE') AS late_bills_count,
    COUNT(*) FILTER (WHERE payment_status = 'UNPAID') AS unpaid_bills_count,
    COALESCE(SUM(amount_dzd) FILTER (WHERE payment_status = 'UNPAID'), 0) AS unpaid_fixe_debt_dzd
FROM fixe_billing
GROUP BY landline_number
ORDER BY landline_number
LIMIT 10;
"""

df_fixe_summary = pd.read_sql(query_fixe, con=engine)
df_fixe_summary

DatabaseError: Execution failed on sql '
SELECT 
    landline_number,
    COUNT(*) AS total_fixe_bills,
    COUNT(*) FILTER (WHERE payment_status = 'LATE') AS late_bills_count,
    COUNT(*) FILTER (WHERE payment_status = 'UNPAID') AS unpaid_bills_count,
    COALESCE(SUM(amount_dzd) FILTER (WHERE payment_status = 'UNPAID'), 0) AS unpaid_fixe_debt_dzd
FROM fixe_billing
GROUP BY landline_number
ORDER BY landline_number
LIMIT 10;
': (psycopg2.errors.UndefinedColumn) column "amount_dzd" does not exist
LINE 7:     COALESCE(SUM(amount_dzd) FILTER (WHERE payment_status = ...
                         ^

[SQL: 
SELECT 
    landline_number,
    COUNT(*) AS total_fixe_bills,
    COUNT(*) FILTER (WHERE payment_status = 'LATE') AS late_bills_count,
    COUNT(*) FILTER (WHERE payment_status = 'UNPAID') AS unpaid_bills_count,
    COALESCE(SUM(amount_dzd) FILTER (WHERE payment_status = 'UNPAID'), 0) AS unpaid_fixe_debt_dzd
FROM fixe_billing
GROUP BY landline_number
ORDER BY landline_number
LIMIT 10;
]


In [9]:
print("fixe_billing columns:", pd.read_sql("SELECT * FROM fixe_billing LIMIT 1;", con=engine).columns.tolist())
print("crm_subscribers columns:", pd.read_sql("SELECT * FROM crm_subscribers LIMIT 1;", con=engine).columns.tolist())

fixe_billing columns: ['invoice_id', 'landline_number', 'billing_cycle', 'fixe_fee_dzd', 'voice_calls_dzd', 'total_due_dzd', 'payment_status', 'paid_date']
crm_subscribers columns: ['subscriber_id', 'landline_number', 'wilaya', 'plan_type', 'speed_mbps', 'monthly_price_dzd', 'install_date', 'modem_model', 'acquisition_channel']


In [10]:
query_fixe = """
SELECT 
    landline_number,
    COUNT(*) AS total_fixe_bills,
    COUNT(*) FILTER (WHERE payment_status = 'LATE') AS late_bills_count,
    COUNT(*) FILTER (WHERE payment_status = 'UNPAID') AS unpaid_bills_count,
    SUM(voice_calls_dzd) AS total_voice_calls_dzd,
    COALESCE(SUM(total_due_dzd) FILTER (WHERE payment_status = 'UNPAID'), 0) AS unpaid_fixe_debt_dzd
FROM fixe_billing
GROUP BY landline_number
ORDER BY landline_number
LIMIT 10;
"""

df_fixe_summary = pd.read_sql(query_fixe, con=engine)
df_fixe_summary

,landline_number,total_fixe_bills,late_bills_count,unpaid_bills_count,total_voice_calls_dzd,unpaid_fixe_debt_dzd
0,021100513,6,1,1,1960,1180
1,021101211,6,0,0,3190,0
2,021103269,6,0,0,1500,0
3,021104694,6,0,2,2710,1830
4,021107543,6,1,0,3080,0
5,021114631,6,1,0,1130,0
6,021120961,6,1,0,2870,0
7,021121317,6,1,2,2700,2200
8,021122040,6,0,0,1670,0
9,021122624,6,0,0,1110,0


In [11]:
query_abt = """
WITH recharges_summary AS (
    SELECT 
        subscriber_id,
        COUNT(*) AS total_tx_attempts,
        COUNT(*) FILTER (WHERE tx_status = 'SUCCESS') AS successful_recharges,
        COUNT(*) FILTER (WHERE tx_status != 'SUCCESS') AS failed_recharges,
        COUNT(*) FILTER (WHERE payment_method IN ('Edahabia_Online', 'CIB_Online')) AS online_payment_attempts,
        SUM(amount_dzd) AS total_recharged_dzd,
        SUM(days_granted) AS total_days_granted,
        MAX(timestamp) FILTER (WHERE tx_status = 'SUCCESS') AS last_successful_recharge_date
    FROM internet_recharges
    GROUP BY subscriber_id
),
tickets_summary AS (
    SELECT 
        subscriber_id,
        COUNT(*) AS total_tickets,
        COUNT(*) FILTER (WHERE closed_at IS NULL) AS unresolved_tickets,
        ROUND(AVG(line_attenuation_db)::numeric, 2) AS avg_attenuation_db,
        MAX(line_attenuation_db) AS max_attenuation_db,
        ROUND(AVG(EXTRACT(EPOCH FROM (closed_at - opened_at)) / 3600.0)::numeric, 2) AS avg_resolution_hours
    FROM technical_tickets
    GROUP BY subscriber_id
),
fixe_summary AS (
    SELECT 
        landline_number,
        COUNT(*) AS total_fixe_bills,
        COUNT(*) FILTER (WHERE payment_status = 'LATE') AS late_bills_count,
        COUNT(*) FILTER (WHERE payment_status = 'UNPAID') AS unpaid_bills_count,
        SUM(voice_calls_dzd) AS total_voice_calls_dzd,
        COALESCE(SUM(total_due_dzd) FILTER (WHERE payment_status = 'UNPAID'), 0) AS unpaid_fixe_debt_dzd
    FROM fixe_billing
    GROUP BY landline_number
)
SELECT 
    sub.subscriber_id,
    sub.landline_number,
    sub.wilaya,
    sub.plan_type,
    sub.speed_mbps,
    sub.monthly_price_dzd,
    sub.install_date,
    sub.modem_model,
    sub.acquisition_channel,
    
    -- ميزات التعبئة
    COALESCE(rech.total_tx_attempts, 0) AS total_tx_attempts,
    COALESCE(rech.successful_recharges, 0) AS successful_recharges,
    COALESCE(rech.failed_recharges, 0) AS failed_recharges,
    COALESCE(rech.online_payment_attempts, 0) AS online_payment_attempts,
    COALESCE(rech.total_recharged_dzd, 0) AS total_recharged_dzd,
    COALESCE(rech.total_days_granted, 0) AS total_days_granted,
    rech.last_successful_recharge_date,
    
    -- ميزات الشكاوى التقنية
    COALESCE(tkt.total_tickets, 0) AS total_tickets,
    COALESCE(tkt.unresolved_tickets, 0) AS unresolved_tickets,
    COALESCE(tkt.avg_attenuation_db, 0) AS avg_attenuation_db,
    COALESCE(tkt.max_attenuation_db, 0) AS max_attenuation_db,
    COALESCE(tkt.avg_resolution_hours, 0) AS avg_resolution_hours,
    
    -- ميزات فواتير الهاتف الثابت
    COALESCE(fix.total_fixe_bills, 0) AS total_fixe_bills,
    COALESCE(fix.late_bills_count, 0) AS late_bills_count,
    COALESCE(fix.unpaid_bills_count, 0) AS unpaid_bills_count,
    COALESCE(fix.total_voice_calls_dzd, 0) AS total_voice_calls_dzd,
    COALESCE(fix.unpaid_fixe_debt_dzd, 0) AS unpaid_fixe_debt_dzd

FROM crm_subscribers sub
LEFT JOIN recharges_summary rech ON sub.subscriber_id = rech.subscriber_id
LEFT JOIN tickets_summary tkt ON sub.subscriber_id = tkt.subscriber_id
LEFT JOIN fixe_summary fix ON sub.landline_number = fix.landline_number
ORDER BY sub.subscriber_id;
"""

df_abt = pd.read_sql(query_abt, con=engine)
print("شكل الجدول الموحد (عدد الأسطر، عدد الأعمدة):", df_abt.shape)
df_abt.head()

شكل الجدول الموحد (عدد الأسطر، عدد الأعمدة): (1500, 26)


,subscriber_id,landline_number,wilaya,plan_type,speed_mbps,monthly_price_dzd,install_date,modem_model,acquisition_channel,total_tx_attempts,...,total_tickets,unresolved_tickets,avg_attenuation_db,max_attenuation_db,avg_resolution_hours,total_fixe_bills,late_bills_count,unpaid_bills_count,total_voice_calls_dzd,unpaid_fixe_debt_dzd
0,SUB-10000,031494990,Constantine,Idoom_ADSL,15,2000,2023-06-08,DLink_DSL2750U,Door_to_Door,6,...,1,0,20.7,20.7,78.08,6,0,2,1350,2200
1,SUB-10001,021550316,Algiers,Idoom_ADSL,15,2000,2022-12-15,DLink_DSL2750U,Online_Portal,13,...,0,0,0.0,0.0,0.00,6,0,0,2880,0
2,SUB-10002,041844867,Oran,Idoom_Fibre,180,2600,2022-01-07,ZTE_F660,Door_to_Door,6,...,2,0,17.5,18.5,42.99,6,0,2,1150,1000
3,SUB-10003,021139423,Algiers,Idoom_Fibre,60,2000,2021-05-23,Nokia_G240W,Door_to_Door,12,...,0,0,0.0,0.0,0.00,6,1,0,1670,0
4,SUB-10004,041216929,Oran,Idoom_Fibre,180,2600,2022-05-28,Nokia_G240W,Commercial_Agency,11,...,0,0,0.0,0.0,0.00,6,1,0,1670,0


In [12]:
# 1. تحويل أعمدة التواريخ إلى صيغة datetime موحدة في Pandas
df_abt["install_date"] = pd.to_datetime(df_abt["install_date"])
df_abt["last_successful_recharge_date"] = pd.to_datetime(df_abt["last_successful_recharge_date"])

In [13]:
# 2. تحديد التاريخ المرجعي (Snapshot Date) كآخر يوم في البيانات (2025-12-31)
snapshot_date = pd.to_datetime("2025-12-31")

In [15]:
# 3. حساب الأقدمية (Tenure) وأيام الانقطاع منذ آخر تعبئة (Recency)
df_abt["tenure_days"] = (snapshot_date - df_abt["install_date"]).dt.days
df_abt["days_since_last_recharge"] = (snapshot_date - df_abt["last_successful_recharge_date"]).dt.days
# إذا كان هناك مشترك لم ينجح في أي تعبئة أبداً (NaN)، نعتبر أيام انقطاعه هي كامل مدة أقدميته
df_abt["days_since_last_recharge"] = df_abt["days_since_last_recharge"].fillna(df_abt["tenure_days"])

In [16]:
# 4. صناعة ميزات النسب السلوكية (Behavioral Ratios)
df_abt["failed_recharge_ratio"] = (
    df_abt["failed_recharges"] / df_abt["total_tx_attempts"]
).fillna(0).round(4)

df_abt["online_payment_ratio"] = (
    df_abt["online_payment_attempts"] / df_abt["total_tx_attempts"]
).fillna(0).round(4)

df_abt["unpaid_bills_ratio"] = (
    df_abt["unpaid_bills_count"] / df_abt["total_fixe_bills"]
).fillna(0).round(4)

In [17]:
# 5. تعريف متغير الهدف (Target Variable: is_churn) بناءً على قاعدة 90 يوم انقطاع
df_abt["is_churn"] = (df_abt["days_since_last_recharge"] > 90).astype(int)

In [18]:
# معاينة توزيع المشتركين المغادرين (1) والباقين (0)
print("توزيع عمود الهدف (is_churn) بالعدد:")
print(df_abt["is_churn"].value_counts())
print("\nنسبة المغادرة (Churn Rate) المئوية:")
print((df_abt["is_churn"].value_counts(normalize=True) * 100).round(2))

توزيع عمود الهدف (is_churn) بالعدد:
is_churn
0    1155
1     345
Name: count, dtype: int64

نسبة المغادرة (Churn Rate) المئوية:
is_churn
0    77.0
1    23.0
Name: proportion, dtype: float64
